This comprehensive code outlines the implementation and evaluation of an Adaptive Neuro-Fuzzy Inference System (ANFIS) model using Python. It includes functionalities for initializing the ANFIS model, processing inputs through the model's layers, adjusting model parameters, and evaluating model performance. The code integrates various Python libraries, including NumPy for numerical operations and Matplotlib for visualization.

### Key Components:

1. **Membership Function Implementation**:
   - `gbellmf(x, params)`: Defines a Gaussian bell-shaped membership function used to calculate the degree of membership for inputs based on specified parameters.

2. **ANFIS Layer Calculations**:
   - Functions like `calculate_output1` through `calculate_output5` implement the sequential processing layers of the ANFIS model, handling operations from fuzzification to defuzzification.

3. **Parameter Optimization**:
   - `get_kalman_data`, `mykalman`, `clear_de_dp`, `calculate_de_do`, `derivative_o_o`, `do4_do3`, `do3_do2`, `update_de_do`, and `update_parameter` functions work together to adjust the ANFIS model's parameters based on the observed data, employing a version of the Kalman filter and gradient descent mechanisms.

4. **Adaptive Step Size Management**:
   - `update_step_size` dynamically adjusts the learning rate based on error trends to improve model training efficiency, utilizing `check_decrease_ss` and `check_increase_ss` to determine when adjustments are necessary.

5. **Model Training and Evaluation**:
   - `myanfis` is the main function that orchestrates the training of the ANFIS model over multiple epochs, optimizing parameters, and calculating RMSE to track performance.
   - `evalmyanfis` applies the trained ANFIS model to input data, generating predictions.

6. **Visualization Functions**:
   - `plot_Nodes` visualizes the connections between nodes in the ANFIS model.
   - `plot_mf` displays the membership functions for each input variable.
   - `plot_predictions` compares the actual output values against the ANFIS predictions.

7. **Performance Metrics Calculation**:
   - `calc_rmse` and `calc_r2` compute the Root Mean Squared Error (RMSE) and R-squared value, respectively, for the model's predictions compared to actual outputs.
   - `plot_r2` visualizes the correlation between actual and predicted outputs along with the R-squared value.

### Workflow:

1. **Model Initialization**: Define the structure of the ANFIS model, including input variables, membership functions, and rules.
2. **Training**: Process inputs through the ANFIS layers, adjust parameters using optimization techniques, and dynamically manage the step size for efficient learning.
3. **Evaluation**: Apply the trained model to test data, calculate performance metrics, and visualize results.
4. **Visualization**: Understand the model's structure and performance through various plots.

### Usage:

This code is used for creating, training, and evaluating an ANFIS model tailored to specific data. It combines fuzzy logic's handling of uncertainty with neural networks' learning capabilities, offering a powerful approach for modeling complex systems. The visualization and performance metrics functions aid in interpreting the model's effectiveness and understanding its behavior.

In [1]:
#pip install numpy matplotlib scikit-fuzzy

In [3]:
import numpy as np
import csv
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
# Custom Gaussian bell-shaped membership function
def gbellmf(x, params):
    a, b, c = params
    return 1 / (1 + ((x - c) / a) ** (2 * b))

The code defines a custom function named `gbellmf` that calculates the value of a Gaussian Bell-shaped membership function for a given input `x` and a set of parameters `params`. This type of function is often used in fuzzy logic systems, where instead of having binary (true/false) outputs, you get degrees of truthiness represented by values between 0 and 1. The Gaussian Bell-shaped membership function is one way to represent such degrees of truthiness, especially when dealing with continuous variables.

Here's a breakdown of the function and its components:

- **Function Name:** `gbellmf` stands for "Gaussian Bell-shaped Membership Function."

- **Parameters:**
  - `x`: The input value for which the membership value is to be calculated.
  - `params`: A tuple or list of three parameters (`a`, `b`, `c`) that define the shape and location of the bell curve. These parameters are:
    - `a`: Controls the width of the bell curve. A smaller `a` results in a wider bell curve.
    - `b`: Determines the slope of the curve. A larger `b` makes the slope steeper, making the transition from 0 to 1 more abrupt.
    - `c`: Defines the center of the bell curve. This is the value of `x` at which the membership function reaches its maximum value of 1.

- **The Formula:** `1 / (1 + ((x - c) / a) ** (2 * b))`
  - This formula computes the membership value for a given `x`. The result is always between 0 and 1, inclusive.
  - The denominator adjusts the shape of the curve based on the distance of `x` from the center `c`, scaled by the width `a`, and modified by the slope parameter `b`. The `**` operator represents exponentiation, so `(2 * b)` is used to control the curvature.
  - The `1 / (1 + ...)` structure ensures that the function's output is smoothly adjusted between 0 and 1, creating a bell-shaped curve when plotted.

This function can be used in various applications of fuzzy logic, such as in fuzzy control systems, where inputs are not simply true or false, but rather have degrees of truthiness that are represented by continuous values. By adjusting the parameters `a`, `b`, and `c`, different shapes of the membership function can be achieved to suit specific needs.

In [ ]:
def calculate_output1(mynet):
    mparams = mynet['mparams']

    for i in range(mynet['ni']):
        for j in range(mynet['mf']):
            ind = mynet['ni'] + i * mynet['mf'] + j

            x = mynet['nodes'][i]
            a = mparams[i * mynet['mf'] + j, 0]
            b = mparams[i * mynet['mf'] + j, 1]
            c = mparams[i * mynet['mf'] + j, 2]

            tmp1 = (x - c) / a
            if tmp1 == 0:
                tmp2 = 0
            else:
                tmp2 = (tmp1 ** 2) ** b
            mynet['nodes'][ind] = 1 / (1 + tmp2)
    mynet['nodes'] = mynet['nodes']
    return mynet

The provided code defines a function named `calculate_output1` that operates on a dictionary `mynet`, which represents a neural network or a fuzzy logic system. The function appears to be part of a larger system for fuzzy logic processing or neural network computation, particularly focusing on membership function calculations similar to the Gaussian bell-shaped membership function discussed earlier. It calculates the output values based on membership functions for each input and updates the `mynet` dictionary with these values. 

Here's a breakdown of what each part of the function does:

- **Input:** 
  - `mynet`: A dictionary containing several keys that represent different components of a network or system. The keys include `'mparams'` for membership function parameters, `'ni'` for the number of inputs, `'mf'` for the number of membership functions per input, `'nodes'` for storing input values and calculated output values.

- **Process:**
  - The function first extracts the membership function parameters (`mparams`) from the `mynet` dictionary.
  - It then iterates over each input (`'ni'`) and each membership function (`'mf'`) associated with that input.
  - For each combination of input and membership function, it calculates the index (`ind`) in the `'nodes'` array where the output of the membership function calculation will be stored.
  - It retrieves the current input value (`x`) from the `'nodes'` array.
  - It then retrieves the parameters (`a`, `b`, `c`) for the current membership function from `mparams`. These parameters are used to calculate the membership value similarly to a Gaussian bell-shaped function but include an adjustment based on the parameter `b` which might alter the shape of the function.
  - The membership value is calculated using a modified formula that accounts for cases where `(x - c) / a` equals zero, to avoid division by zero errors. Otherwise, it calculates a value `tmp2` which adjusts based on the square of `tmp1` raised to the power of `b`.
  - The calculated membership value is then stored in the `'nodes'` array at the previously calculated index (`ind`).
  - After updating the `'nodes'` array with all the membership values, the function doesn't modify the `'nodes'` array anymore but redundantly assigns it to itself.
  
- **Output:**
  - Returns the updated `mynet` dictionary with the `'nodes'` array now containing the calculated membership values for each input and membership function combination.

This function's role is likely in the context of fuzzy logic controllers or neural networks where the membership values of inputs are crucial for determining the output or decision based on fuzzy rules or further neural network processing. The specific use of `a`, `b`, and `c` parameters suggests the flexibility in shaping the membership function curves for various inputs, which is essential for fine-tuning the system's behavior.


In [ ]:
def calculate_output2(mynet):
    st = mynet['ni'] + mynet['ni'] * mynet['mf']
    for i in range(st, st + mynet['nc']):
        I = [idx for idx, val in enumerate(mynet['config'][:, i]) if val == 1]
        tmp = 1
        for idx in I:
            tmp *= mynet['nodes'][idx]
        mynet['nodes'][i] = tmp
    return mynet

The `calculate_output2` function is another component of a system that seems to manipulate a network or a computational model, similar in context to the previous function you asked about. This function specifically calculates the output for a certain layer within a model, likely based on inputs processed through membership functions or other preliminary computations. It updates the `mynet` dictionary with new values computed during its execution.

Here's a step-by-step breakdown of the function:

- **Input:**
  - `mynet`: A dictionary that includes various keys representing the structure and state of a network or model. The keys referenced in this function are `'ni'` (number of inputs), `'mf'` (number of membership functions per input), `'nc'` (number of configurations or connections for the current layer), `'config'` (a configuration matrix that defines the connections), and `'nodes'` (an array that stores the inputs, outputs, and intermediate calculation results).

- **Process:**
  - The function calculates the starting index `st` for the current layer's outputs in the `'nodes'` array. This is done by adding the number of inputs (`'ni'`) to the product of the number of inputs and the number of membership functions per input (`'ni' * 'mf'`), essentially skipping over the sections of the array used for inputs and their corresponding membership function outputs.
  - It then iterates over a range from this starting index `st` to `st` + the number of configurations/connections for the current layer (`'nc'`), which likely represents each node or output in this layer.
  - For each node in the current layer, it finds the indices (`I`) of all preceding nodes (in `'nodes'`) that are connected to it, as defined by the `'config'` matrix. A connection is indicated by a value of `1` in the `'config'` matrix.
  - It initializes a temporary variable `tmp` to `1`, which will be used to accumulate the product of the values of all connected nodes (found in the previous step).
  - For each of the connected nodes' indices (`idx` in `I`), it multiplies `tmp` by the value of that node in the `'nodes'` array, effectively calculating the product of all inputs coming into the current node.
  - This product (`tmp`) is then assigned as the new value of the current node in the `'nodes'` array.
  
- **Output:**
  - Returns the updated `mynet` dictionary, now with the `'nodes'` array updated to include the calculated values for the nodes in the current layer.

The function seems to be part of a larger system that could be implementing a type of neural network or fuzzy logic controller where the output of each node in a layer is determined by the product of the values of its connected nodes from a previous layer. This could be part of a forward pass in a computational graph, where values are propagated forward based on the network's configuration. The use of a product rule suggests that this particular layer may implement a form of logical "AND" operation across its inputs, which is a common approach in certain types of fuzzy logic systems.

In [ ]:
def calculate_output3(mynet):
    st = mynet['ni'] + mynet['ni'] * mynet['mf'] + mynet['nc']    
    for i in range(st , st + mynet['nc'] ):
        I = [idx for idx, val in enumerate(mynet['config'][:, i]) if val == 1]
        denom = sum([mynet['nodes'][idx] for idx in I])
        mynet['nodes'][i] = mynet['nodes'][i - mynet['nc']] / denom
    return mynet

The `calculate_output3` function is designed to compute and update the outputs for a specific layer or stage in a network, which is represented by the `mynet` dictionary. This function seems to follow the pattern of manipulating a structured network model, possibly for purposes such as fuzzy logic control, neural network computation, or another system that involves a series of computational layers or steps.

Let's break down the function:

- **Input:**
  - `mynet`: A dictionary that contains various keys detailing the structure and state of a network or computational model. Key elements used in this function include `'ni'` (number of inputs), `'mf'` (membership functions per input), `'nc'` (number of configurations or connections for a layer), `'config'` (a configuration matrix that defines connections between nodes), and `'nodes'` (an array storing input, output, and intermediate calculation results).

- **Process:**
  - The function calculates a starting index `st` that marks the beginning of a specific layer's outputs within the `'nodes'` array. This is determined by skipping past sections of the `'nodes'` array that are allocated for inputs and their membership function outputs, as well as the outputs from a previous layer.
  - It iterates over a range from `st` to `st + mynet['nc']`, which likely corresponds to each output node in this particular layer of the network.
  - For each node in the current layer, it identifies the indices (`I`) of all connected nodes from a previous layer, based on the `'config'` matrix. A connection is indicated by a value of `1`.
  - The function then calculates the denominator (`denom`) as the sum of the values of all the connected nodes identified in the previous step.
  - It updates the value of the current node in the `'nodes'` array by dividing the node's previous value (from the immediately preceding layer, as indicated by `i - mynet['nc']`) by the calculated `denom`. This operation might represent a normalization step or a specific type of calculation based on the ratios of node values.
  
- **Output:**
  - Returns the updated `mynet` dictionary with the `'nodes'` array now containing the newly calculated values for the nodes in the current layer.

The logic of `calculate_output3` suggests a focus on adjusting the outputs of a network layer based on the sum of certain inputs or previous outputs, possibly to normalize or scale these values in the context of the overall system's computation. This could be part of a normalization step in a fuzzy logic system, a specific neural network layer operation, or another computational model that requires adjusting values based on a sum or ratio of inputs. This approach is indicative of systems where the propagation of values through the network involves complex interactions beyond simple weighted sums, possibly incorporating normalization or scaling to ensure outputs remain within a certain range or to emphasize the relative importance of different inputs.

In [ ]:
def calculate_output4(mynet):
    st = mynet['ni'] + mynet['ni'] * mynet['mf'] + 2 * mynet['nc']
    inp = mynet['nodes'][:mynet['ni']]
    kparam = mynet['kparams']
    
    for i in range(mynet['nc']):
        wn = mynet['nodes'][i + st - mynet['nc']]
        mynet['nodes'][i + st] = wn * (np.sum(kparam[i, :-1] * inp) + kparam[i, -1])
    
    # print(mynet['nodes'])
    # print(np.shape(mynet['nodes']))
    # exit()
    return mynet

The `calculate_output4` function is part of a computational system or model, as suggested by the consistent use of the `mynet` dictionary across the previous functions you've inquired about. This particular function seems to implement a specific calculation step within a network, potentially within a neural network, fuzzy logic controller, or another modular computational system. Let's dissect the components and functionality of the code:

- **Input:**
  - `mynet`: A dictionary encapsulating the network's structure and state, including keys for the number of inputs (`'ni'`), the number of membership functions per input (`'mf'`), the number of configurations or connections (`'nc'`), an array of node values (`'nodes'`), and a matrix of parameters (`'kparams'`).

- **Process:**
  - The starting index `st` is calculated to determine where in the `'nodes'` array the function should begin updating values. This index is computed to skip past inputs, their membership function outputs, and two sets of configuration/connection outputs, indicating this function operates on a subsequent layer or stage within the network.
  - The input values (`inp`) are extracted from the beginning of the `'nodes'` array, up to the number of inputs (`'ni'`).
  - The parameter matrix `kparam` is retrieved from `mynet`, which likely contains weights or coefficients used in the calculation for this layer.
  - The function iterates over the range of `'nc'` (the number of configurations or connections for this layer), performing a calculation for each node in this layer. For each node:
    - It retrieves the weight or coefficient (`wn`) from a previous step's output in the `'nodes'` array.
    - The new value for the node is calculated by multiplying `wn` with the sum of the product of `kparam` coefficients for this node (excluding the last coefficient) and the input values, then adding the last coefficient from `kparam`. This resembles a weighted sum operation, a common computation in neural networks and other computational models, where the last coefficient might act as a bias term.
  
- **Output:**
  - The function updates the `'nodes'` array within the `mynet` dictionary with the new calculated values and returns the modified `mynet`.

This function's operations hint at a layer within a computational system where each node's output is determined by a combination of weighted inputs (possibly from an earlier layer or the original input) and a bias, with the weight (`wn`) from a previous calculation step factoring into the new value. This pattern is typical in neural networks where each neuron's output is a function of its inputs, weights, and bias, suggesting that `calculate_output4` might be implementing a layer of a neural network, with specific details tailored to the overall architecture and purpose of the `mynet` system.

In [ ]:
def calculate_output5(mynet):
    mynet['nodes'][-1] = sum(mynet['nodes'][-mynet['nc']-1:-1])
    # print(np.shape(mynet['nodes']))
    # print(mynet['nodes'])
    # exit()
    return mynet


The `calculate_output5` function performs a simple yet specific operation on the `mynet` dictionary, particularly updating the last element in the `mynet['nodes']` array. This function likely represents a concluding step in a sequence of computations within a network model, such as a neural network, fuzzy logic system, or another computational framework that processes inputs through multiple layers or stages.

Here's a breakdown of the function:

- **Input:**
  - `mynet`: A dictionary that encapsulates the state and structure of a computational model or network. Key to this function is the `'nodes'` array, which stores values representing either raw inputs, intermediate computations, or outputs from various stages of the network, and `'nc'`, which denotes the number of configurations or connections at a certain stage in the network.

- **Process:**
  - The function updates the last element of the `mynet['nodes']` array with the sum of a specific subset of the array. This subset is defined by the slice `[-mynet['nc']-1:-1]`, which targets the elements from the second-to-last set of `nc` elements up to, but not including, the last element. Essentially, it sums the values of the nodes from a previous layer or stage in the network.
  - This operation could be seen as aggregating the outcomes of the last set of computations (such as the outputs of the last layer of neurons in a neural network, or the last set of rules in a fuzzy logic controller) to produce a single final output value.

- **Output:**
  - Returns the updated `mynet` dictionary, now with the last element of the `'nodes'` array reflecting the sum of the selected previous outputs.

The purpose of `calculate_output5` might be to consolidate the results of a complex computation into a single output value, which is a common requirement in systems designed to aggregate multiple inputs or intermediate results into a final decision, score, or evaluation metric. This could be useful in scenarios where the overall outcome of the network's processing needs to be summarized or where the final layer's outputs are combined to produce a single result for further analysis, decision-making, or as the end result of a predictive model.

In [ ]:
def get_kalman_data(mynet, target):
    
    kalman_data = np.zeros(((mynet['ni'] + 1) * mynet['nc'] + 1, 1))
    
    st = mynet['ni'] + mynet['ni'] * mynet['mf'] + mynet['nc']
    j = 0
    # print(st + mynet['nc'] + 1)
    for i in range(st, st + mynet['nc']):
        for k in range(mynet['ni']):
            kalman_data[j] = mynet['nodes'][i] * mynet['nodes'][k]
            j += 1
        kalman_data[j] = mynet['nodes'][i]
        j += 1

    kalman_data[j] = target
    return kalman_data

The `get_kalman_data` function is designed to extract and structure data from the `mynet` dictionary in a way that's conducive to Kalman filter processing or a related algorithmic use, focusing on a system involving a network or computational model. The function processes specific nodes' values from the `mynet` structure, combines them in a specific manner, and appends a target value, preparing a dataset possibly for predictive modeling or state estimation tasks. Let's dissect this function:

- **Input:**
  - `mynet`: A dictionary containing the structure and state of a computational model or network, which includes information such as the number of inputs (`'ni'`), the number of membership functions per input (`'mf'`), the number of configurations or connections (`'nc'`), and an array of node values (`'nodes'`).
  - `target`: A variable representing the target value to be appended at the end of the constructed data array. This could be a value that the network aims to predict or estimate, or it may serve as a reference for algorithmic processing.

- **Process:**
  - The function initializes a numpy array `kalman_data` with zeros. The size of this array is calculated based on the number of inputs (`ni`), the number of configurations or connections (`nc`), plus an additional slot for the target value, making the total size `((mynet['ni'] + 1) * mynet['nc'] + 1, 1)`. This structure suggests that for each configuration/connection, the function aims to store a value for every input multiplied by a specific node's value, plus the node's value itself, and finally the target value.
  - It calculates a starting index `st` which seems to skip over inputs and their membership function outputs, as well as a first layer of configurations/connections, pointing to a specific layer or stage in the network for processing.
  - Through a nested loop, for each configuration/connection in the specified layer (`range(st, st + mynet['nc'])`), it multiplies each node's value at this stage by the values of each input node and stores these products sequentially in `kalman_data`. Additionally, for each configuration/connection, after processing all inputs, it stores the value of the current node itself in the array.
  - After filling the array with these calculated values and the nodes' own values, the final slot in `kalman_data` is set to the `target` value.

- **Output:**
  - Returns the `kalman_data` array, which now contains a structured dataset ready for further processing, potentially with a Kalman filter or a similar algorithm.

This function is particularly interesting because it reflects a process of data preparation that combines network outputs and inputs in a manner that suggests a form of predictive modeling or state estimation, where interactions between different network parts and an external target value are significant. The inclusion of the target value at the end implies that the resulting dataset may be used for supervised learning tasks, such as training or refining a model to predict or estimate this target based on the network's internal states and inputs.

In [ ]:
def mykalman(mynet, kalman_data, k):
    k_p_n = (mynet['ni'] + 1) * mynet['nc']

    alpha = 1000000
    # print(k)
    if k == 0:
        mynet['P'] = np.zeros((k_p_n, 1))
        mynet['S'] = alpha * np.eye(k_p_n)
    
    # print(mynet['P'])
    # print(mynet['S'])
        
    # exit()
    x = kalman_data[:-1]
    y = kalman_data[-1]
           
    
    x_temp = x.flatten()
    tmp1 = np.dot(x_temp, mynet['S'])
    tmp1 = tmp1[:, np.newaxis]

    denom = 1 + np.sum(tmp1 * x)
    tmp1 = np.dot(mynet['S'], x_temp)
    tmp1 = tmp1[:, np.newaxis]
        

    tmp2 = np.dot(x_temp, mynet['S'])
    tmp2 = tmp2[:, np.newaxis]
    tmp_m = np.outer(tmp1, tmp2)
    tmp_m = -1 / denom * tmp_m
    mynet['S'] = mynet['S'] + tmp_m
    
    diff = y - np.sum(x * mynet['P'])
    tmp1 = diff * np.dot(mynet['S'], x_temp)
    tmp1 = tmp1[:, np.newaxis]
    
    mynet['P'] = mynet['P'] + tmp1
    mynet['kparams'] = mynet['P'].reshape(( mynet['nc'],mynet['ni'] + 1))
    # mynet['kparams'] = mynet['P'].reshape(8,4)

    return mynet

The `mykalman` function appears to implement a variation of the Kalman filter update algorithm, tailored to work within a specific system represented by the `mynet` dictionary. The Kalman filter is a recursive algorithm used for estimating the state of a linear dynamic system from a series of noisy measurements. It's widely used in control systems, for navigation and tracking, and in finance, among other fields. This particular implementation updates `mynet` with new estimates based on incoming data (`kalman_data`) and the current step (`k`).

Here's a simplified explanation of the key components and steps in the function:

### Initialization
- `k_p_n`: A variable representing the total number of parameters to estimate, calculated based on the network's configuration.
- `alpha`: A large number used to initialize the covariance matrix (`S`) of the estimate with high uncertainty. This is a common practice in Kalman filters when the true variance of the state is unknown.
- When `k` (the current iteration or time step) is 0, it initializes two matrices in `mynet`:
  - `P`: Predicted state estimate matrix, initialized to zeros.
  - `S`: Covariance matrix of the estimate, initialized to be a scaled identity matrix (`alpha` times an identity matrix), indicating initial uncertainty in the estimates.

### State and Measurement Update
- It separates the `kalman_data` into:
  - `x`: The state vector (all elements except the last), representing the current observations or measurements.
  - `y`: The measurement corresponding to the current state (the last element of `kalman_data`).
- The function then proceeds to update the covariance matrix (`S`) and the predicted state (`P`) based on the new measurements (`x` and `y`), following the Kalman filter update equations.

### Kalman Gain Calculation and Update
- It computes a temporary variable `tmp1` for intermediate calculations and updates the covariance matrix (`S`) with `tmp_m`, which is derived from Kalman gain calculations.
- The difference between the actual measurement `y` and the estimated measurement (from `x` and `P`) is calculated as `diff`.
- It updates the predicted state estimate (`P`) with this new information.

### Reshape and Return
- Finally, it reshapes the updated predicted state estimate `P` into `kparams` based on the network's configurations. This operation suggests that `P` contains updated parameters for the system modeled by `mynet`, tailored for each configuration/connection and input.
- The updated `mynet` dictionary is returned, now containing the latest estimates and covariance matrix after processing the current measurements.

### Summary
In essence, this function uses a series of measurements (encapsulated in `kalman_data`) to update the estimates of a system's state (`P`) and the uncertainty of those estimates (`S`). It's a customized implementation of a Kalman filter update, integrated within a specific system model (`mynet`). The use of Kalman filtering techniques here implies an attempt to improve the accuracy of the system's parameters or state estimates over time, taking into account the latest observations and correcting for any discrepancies between predicted and actual measurements.

In [ ]:
def clear_de_dp(mynet):
    mynet['mparam_de_do'] = np.zeros((mynet['ni'] * mynet['mf'], 3))
    mynet['kparam_de_do'] = np.zeros((mynet['nc'], mynet['ni'] + 1))

    return mynet

This function, `clear_de_dp`, is designed to reset specific elements within a dictionary named `mynet`, which seems to represent a network or model in a computational system, possibly within the context of machine learning, neural networks, or fuzzy logic controllers. The function sets two items in the dictionary, `mparam_de_do` and `kparam_de_do`, to zero matrices of specified dimensions, effectively clearing any previous values they might have held.

Here's a detailed breakdown:

- **Input:** 
  - `mynet`: A dictionary that likely contains various parameters and configurations of a computational model or network. This input allows the function to modify the state of `mynet` directly.

- **Process:**
  - `mynet['mparam_de_do'] = np.zeros((mynet['ni'] * mynet['mf'], 3))`: This line initializes a zero matrix with dimensions `(mynet['ni'] * mynet['mf'], 3)`. The `mynet['ni']` likely stands for the number of inputs to the network, and `mynet['mf']` for the number of membership functions per input. This matrix, `mparam_de_do`, is thus resized to accommodate all input-membership function pairs across its rows, with three columns for storing data. The specific use of three columns suggests that each row might hold parameters or gradients necessary for a particular computation, such as error derivatives with respect to output for each input-membership function pair.
  
- `mynet['kparam_de_do'] = np.zeros((mynet['nc'], mynet['ni'] + 1))`: This line creates a zero matrix with dimensions `(mynet['nc'], mynet['ni'] + 1)`. Here, `mynet['nc']` likely denotes the number of configurations or connections in a particular layer of the network. Each row in this matrix could correspond to a configuration/connection, with columns for each input plus an additional one, possibly for a bias term or an extra parameter. The initialization to zeros suggests resetting or preparing for a new computation, such as updating weights or gradients during an optimization process.

- **Output:** 
  - The function returns the modified `mynet` dictionary, now with the `mparam_de_do` and `kparam_de_do` fields reset to the newly initialized zero matrices.

**Purpose and Context:**
The function's naming convention (`clear_de_dp`) and its operations hint at its role in preparing or resetting part of a network's state, particularly related to gradients or derivatives (`de_do` could stand for derivative of error with respect to outputs). This is a common requirement in optimization routines, such as backpropagation in neural networks, where gradients with respect to parameters are computed during each iteration of training to update the parameters in the direction that minimizes the error. Resetting these matrices to zeros is crucial at the start of such a computation to ensure that new gradients are not contaminated by values from previous iterations.

In [ ]:
def calculate_de_do(mynet, de_dout):
    mynet['de_do'] = np.zeros_like(mynet['nodes'])
    mynet['de_do'][-1] = de_dout
    tmp2 = []

    for i in range(len(mynet['nodes']) - 2, mynet['ni']-1, -1):
        # print("i = ", i)
        # print(len(mynet['nodes']) - 2)
        # print(mynet['ni']+1)
        
        de_do = 0
        II = np.where(mynet['config'][i, :] == 1)[0]
        # print(mynet['config'][i, :])
        # print("II =", II)
        I = np.where(II > i)[0]
        # print("I =",I)      
        for j in I:
            jj = II[j]
            tmp1 = mynet['de_do'][jj]            
            tmp2 = derivative_o_o(mynet, i, jj) 
            # print("tmp2 =",tmp2)
            # ui = input()            
            de_do = de_do + tmp1 * tmp2
        mynet['de_do'][i] = de_do
        # print(tmp2)
        # ui = input()
        # print(I, mynet['de_do'][i], tmp1, tmp2)
        # exit()
    # print(np.shape(mynet['de_do']))
    # print((mynet['de_do']))
    # exit()
    return mynet

The `calculate_de_do` function is designed to calculate and update a derivative-related property (`de_do`) within the `mynet` dictionary, which seems to represent a computational model or network structure. Specifically, this function appears to be part of a larger framework, possibly related to training neural networks or other gradient-based optimization tasks, where `de_do` represents the derivative of the error with respect to the outputs of the network nodes.

### Key Components and Steps:

- **Input Parameters:**
  - `mynet`: A dictionary representing the network, containing information about its nodes, configurations, and possibly other parameters relevant to its operation.
  - `de_dout`: A scalar or array representing the derivative of the error with respect to the output of the last node or a specific target value.

- **Initialization:**
  - The function initializes `mynet['de_do']` to a zero array of the same shape as `mynet['nodes']`, which likely stores the values or activations of the network's nodes.
  - It then sets the last element of `mynet['de_do']` to `de_dout`, effectively seeding the backpropagation of errors from the output back through the network.

- **Backpropagation Loop:**
  - The function iterates backward through the nodes of the network (excluding input nodes, hence the stopping condition at `mynet['ni']-1`), calculating the derivative of the error with respect to each node's output (`de_do`).
  - For each node `i` in this reverse iteration, the function:
    - Finds indices (`II`) of nodes that are configured as outputs from the current node (`i`) based on `mynet['config']`, which appears to define the network's topology or connectivity.
    - Filters (`I`) these indices to only include those nodes that come after the current node (`i`), implying a forward connection in the network topology.
    - Accumulates (`de_do`) contributions from these connected nodes, weighted by the derivative of the output of node `jj` with respect to the output of node `i` (`tmp2`). This is computed by `derivative_o_o(mynet, i, jj)`, a function presumably calculating the partial derivative of the output of node `jj` with respect to the output of node `i`, factoring in the nature of the connection or the function relating these nodes.
  
- **Updating `mynet`:**
  - The derivative of the error with respect to the output of each node (`de_do`) is updated in `mynet['de_do']` based on the accumulated contributions.

- **Return Value:**
  - The function returns the updated `mynet` dictionary, now including the newly calculated `de_do` values for backward propagation.

### Purpose and Context:

This function is critical in the context of gradient-based learning or optimization, where understanding how changes in the outputs of each node affect the overall error is necessary to update model parameters effectively (e.g., weights in a neural network). The calculated derivatives are used in the backpropagation algorithm to adjust the parameters in a direction that minimally reduces the error, thereby improving the model's performance iteratively.

The presence of this function hints at a custom implementation of a learning algorithm, potentially for a network with a non-standard architecture or a specific application requiring detailed control over the error backpropagation process.

In [ ]:
def derivative_o_o(mynet, i, j):
    if i > mynet['ni'] + mynet['ni'] * mynet['mf'] +  2 * mynet['nc']-1:
        # print(i, "\t In 1", mynet['ni'] + mynet['ni'] * mynet['mf'] + 2 * mynet['nc']-1)
        return 1
    elif i > mynet['ni'] + mynet['ni'] * mynet['mf'] + mynet['nc']-1:
        # print(i, "\t In 2", mynet['ni'] + mynet['ni'] * mynet['mf'] + mynet['nc'])
        return do4_do3(mynet, i, j)
    elif i > mynet['ni'] + mynet['ni'] * mynet['mf']-1:
        # print(i, "\t In 3", mynet['ni'] + mynet['ni'] * mynet['mf'])
        return do3_do2(mynet, i, j)
    elif i > mynet['ni']-1:
        # print(i, "\t In 4",mynet['ni'])
        # print(mynet['nodes'][j])
        # print(mynet['nodes'][i])
        return mynet['nodes'][j] / mynet['nodes'][i]

The `derivative_o_o` function calculates the derivative of the output of one node with respect to another within a network, based on their positions (`i` and `j`) in the network and its configuration. This is typically used in gradient-based optimization processes, such as training neural networks, where understanding how changes in one part of the network affect another is crucial for adjusting parameters effectively.

### Breakdown of the Function:

- **Parameters:**
  - `mynet`: A dictionary containing the network's configuration, including the number of inputs (`'ni'`), membership functions per input (`'mf'`), and connections (`'nc'`).
  - `i`, `j`: Indices of the nodes for which the derivative is to be calculated, where `i` is the index of the node whose output is being differentiated with respect to, and `j` is the index of the node contributing to the output.

- **Process:**
  - The function first checks the position of node `i` relative to various calculated thresholds that segment the network into different layers or functional groups. These thresholds are based on the structure of the network, indicated by the number of inputs, membership functions per input, and connections.
  
  - **If Conditions:**
    - **First Condition:** If `i` is beyond the last segment of the network (`i > mynet['ni'] + mynet['ni'] * mynet['mf'] +  2 * mynet['nc']-1`), the derivative is considered to be 1. This implies that the output of nodes in this segment does not change with respect to other nodes' outputs, suggesting these nodes may represent a final output layer or an external input where the derivative is direct and does not depend on internal network dynamics.
    
    - **Second Condition:** If `i` falls within a specific range, it calls `do4_do3(mynet, i, j)`, implying a calculation specific to the derivative between two defined stages or layers within the network (possibly `do4` and `do3` are placeholders for specific functional descriptions of these layers).
    
    - **Third Condition:** Similarly, if `i` is within another range, it calls `do3_do2(mynet, i, j)`, indicating another layer-specific derivative calculation between different stages or types of nodes.
    
    - **Fourth Condition:** For nodes beyond the input layer but within the first computational or membership function layer (`i > mynet['ni']-1`), it calculates the derivative as the ratio of the output of node `j` to the output of node `i`. This suggests a direct relationship between these nodes' outputs, possibly reflecting a basic computational or transformation step within the network.

- **Purpose:**
  - This function is designed to dynamically calculate the derivative between outputs of different nodes based on their roles within the network. By adjusting how the derivative is calculated based on node position, the function supports a network with a complex or layered structure, where different types of nodes or layers perform different functions and thus have different relationships between their outputs.

### Conclusion:

`derivative_o_o` is a critical component for implementing backpropagation or other gradient-based optimization methods in a custom computational model. By specifying how the output of one node affects another across different segments of the network, it facilitates precise adjustments to the model's parameters during training or optimization.

In [ ]:
def do4_do3(mynet, i, j):
    kparam = mynet['kparams']
    inp = mynet['nodes'][:mynet['ni']]
    # print("inp =", inp)    
    jj = j - mynet['ni'] - mynet['ni'] * mynet['mf'] - 2 * mynet['nc']   
    # print("jj =", jj)  
    #print("inp and jj - ", inp,jj )
    return np.sum(kparam[jj, :-1] * inp) + kparam[jj, -1]

The `do4_do3` function calculates the derivative of the output between two specific layers or stages within the network represented by the `mynet` dictionary. It seems tailored for a network where the nodes and their connections are structured in such a way that different layers perform distinct computations. Here, `do4_do3` likely represents the derivative of the output of a node in layer 4 with respect to the output of a node in layer 3, indicating a specific transformation or relationship between these two layers.

### Understanding the Code:

- **Input Parameters:**
  - `mynet`: A dictionary that encapsulates the network's configuration and state, including the parameters (`'kparams'`) and the values or outputs of nodes (`'nodes'`).
  - `i`, `j`: Indices representing specific nodes within the network, with `i` and `j` indicating nodes in layer 4 and layer 3, respectively (based on the function's naming convention).

- **Key Operations:**
  - **Extracting Network Parameters and Inputs:**
    - `kparam`: Retrieves the parameter matrix from `mynet`, which contains the weights or coefficients for the network's connections.
    - `inp`: Extracts the input values from the beginning of the `'nodes'` array, up to the number of inputs (`'ni'`). These inputs are presumably the raw inputs to the network.
  
  - **Adjusting Index for `kparam`:**
    - The calculation of `jj` adjusts the index `j` to correctly reference the relevant row in `kparam`. This adjustment accounts for the structure of the network, offsetting `j` by the sum of inputs, the product of inputs and membership functions, and twice the number of configurations/connections. This suggests that `jj` is aimed at isolating a specific subset or layer within the network parameters that corresponds to the transformation from layer 3 to layer 4.

- **Derivative Calculation:**
  - The function calculates a value based on the adjusted index `jj` and the input values. Specifically, it multiplies each input value by its corresponding weight in `kparam` (excluding the last weight, which is presumably a bias term) and sums these products. The bias term (`kparam[jj, -1]`) is then added to this sum. This operation likely represents the derivative of the network's output with respect to a particular node's output, taking into account the weighted sum of inputs and a bias term, which is a common computation in neural networks.

### Purpose and Context:

The `do4_do3` function is particularly useful in contexts where precise control over the transformation between layers is necessary, such as in the training of neural networks through backpropagation, where gradients or derivatives of outputs with respect to inputs and parameters are crucial. By computing how changes in layer 3 affect changes in layer 4 through the specific parameters (`kparams`), this function facilitates the update of network parameters in a way that minimizes error or optimizes some performance metric.

This function, by focusing on the relationship between two consecutive layers within a structured network, underscores the modular or layered approach often employed in neural network design, allowing for the isolation and optimization of specific parts of the network based on their functional contributions to the overall computation.

In [ ]:
def do3_do2(mynet, i, j):
    II = np.where(mynet['config'][:, j] == 1)[0]
    I = np.where(II < j)[0]
    total = np.sum(mynet['nodes'][II[I]])
    # print("total = ",total)
    # print("j = ",j)
    # print("mynet['nc'] = ",mynet['nc'])
    if j - i == mynet['nc']:
        # print(("r1 = ",total - mynet['nodes'][i]) / (total**2))
        return (total - mynet['nodes'][i]) / (total**2)
        
    else:
        # print("r2 = ",-mynet['nodes'][j - mynet['nc']] / (total**2))
        return -mynet['nodes'][j - mynet['nc']] / (total**2)

The `do3_do2` function is part of a larger computational framework, likely aimed at modeling or analyzing a network (as represented by the `mynet` dictionary). This function specifically calculates the derivative of the output of nodes in one layer (presumably layer 3) with respect to the nodes in a preceding layer (presumably layer 2), based on the network's structure and the values of its nodes.

### Key Components:

- **Input Parameters:**
  - `mynet`: A dictionary containing the network's configuration, such as its connectivity (`'config'`) and the values or states of its nodes (`'nodes'`).
  - `i`, `j`: Indices representing specific nodes in the network, where `i` is a node in layer 2 and `j` is a node in layer 3.

- **Process Explained:**
  - **Finding Connected Nodes:** The function begins by identifying nodes that are connected to node `j` (in layer 3), using the network's configuration matrix (`mynet['config']`). It looks for all nodes that have a direct connection to node `j` (`II = np.where(mynet['config'][:, j] == 1)[0]`), and then filters these to find only those nodes that precede node `j` in the network (`I = np.where(II < j)[0]`). This step effectively isolates the relevant inputs or influences on node `j` from the previous layer (layer 2).
  
  - **Calculating Total Influence:** It calculates the sum of the values of these connected nodes (`total = np.sum(mynet['nodes'][II[I]])`), representing the aggregated influence or input from layer 2 to node `j` in layer 3.
  
  - **Derivative Calculation:** The function then calculates the derivative based on the relative positions of nodes `i` and `j`, and their relation within the network:
    - If `j - i` equals the number of configurations/connections (`mynet['nc']`), it suggests a direct sequential relationship between nodes `i` and `j`. The derivative in this case is calculated as `(total - mynet['nodes'][i]) / (total**2)`, indicating how the change in the output of node `i` affects node `j`, taking into account the total influence on node `j`.
    - Otherwise, the derivative is calculated as `-mynet['nodes'][j - mynet['nc']] / (total**2)`. This case seems to address indirect or less straightforward interactions between nodes `i` and `j`, using a different approach to quantify the relationship.

### Purpose and Context:

The `do3_do2` function plays a crucial role in the analysis or training of the network, especially in algorithms that require understanding how changes in one part of the network affect another. Specifically, this could be part of a backpropagation algorithm in neural network training, where calculating the derivative of the output with respect to earlier layers' outputs is essential for updating weights and biases.

This derivative calculation is vital for optimizing the network's performance, as it helps in adjusting the network's parameters to reduce error or improve accuracy in predictive tasks. By quantifying the influence of layer 2 on layer 3, `do3_do2` contributes to a nuanced understanding of the network's internal dynamics, facilitating targeted adjustments that enhance overall functionality.

In [ ]:
def update_de_do(mynet):
    s = 0
    for i in range(mynet['ni'], mynet['ni'] + mynet['ni'] * mynet['mf']):
        for j in range(3):
            do_dp = dmf_dp(mynet, i, j)
            mynet['mparam_de_do'][s, j] = mynet['mparam_de_do'][s, j] + mynet['de_do'][i] * do_dp
        s += 1

    s = 0
    for i in range(1 + mynet['ni'] + mynet['ni'] * mynet['mf'] + 2 * mynet['nc'], len(mynet['config'])):
        for j in range(mynet['ni'] + 1):
            do_dp = dconsequent_dp(mynet, i, j)
            mynet['kparam_de_do'][s, j] = mynet['kparam_de_do'][s, j] + mynet['de_do'][i] * do_dp
        s += 1

    return mynet

The `update_de_do` function updates derivatives within the `mynet` dictionary, a structure representing a computational model or network. This function focuses on two types of parameters: those associated with membership functions (`mparam_de_do`) and those related to the network's consequent parameters (`kparam_de_do`). It calculates the partial derivatives of the error with respect to these parameters and updates the respective matrices in `mynet`.

### Breakdown of the Function:

#### First Loop (Updating `mparam_de_do`):
- Iterates over a range determined by the number of inputs (`ni`) and the number of membership functions per input (`mf`). This suggests the function is dealing with a fuzzy logic controller or a neural network that employs fuzzy logic for its inputs.
- For each input-membership function pair, it iterates three times, corresponding to the typical number of parameters in a triangular or Gaussian membership function (e.g., center, width, and possibly slope or shape).
- `do_dp = dmf_dp(mynet, i, j)`: Calls another function (`dmf_dp`) to calculate the derivative of the output with respect to each parameter of the membership function. The exact nature of `dmf_dp` isn't shown, but it presumably computes how small changes in membership function parameters affect the output of the network or a specific layer.
- Updates `mparam_de_do` by adding the product of the derivative of the error with respect to the output (`de_do[i]`) and the derivative of the output with respect to the parameter (`do_dp`). This effectively accumulates the gradient for each membership function parameter, facilitating gradient-based optimization.

#### Second Loop (Updating `kparam_de_do`):
- Iterates over indices starting from a point beyond the inputs and membership functions, likely targeting the parameters of the rules or the consequent part of a fuzzy system, or the weights in a neural network layer.
- For each parameter, it calculates `do_dp = dconsequent_dp(mynet, i, j)`, which is the derivative of the output with respect to each consequent parameter. Again, the details of `dconsequent_dp` aren't provided but it would similarly compute the sensitivity of the network's output to changes in these parameters.
- Updates `kparam_de_do` similarly by adding the product of the derivative of the error with respect to the network's output and the derivative of the output with respect to the consequent parameter.

#### Functionality and Purpose:
- This function plays a critical role in learning and optimization processes within the network. By calculating and updating the gradients of error with respect to various parameters, it sets the stage for adjusting these parameters to minimize error, following the principle of gradient descent or similar optimization algorithms.
- The structure and logic of the function suggest its application in a system that combines fuzzy logic with neural networks, or a purely fuzzy system, where both the membership function parameters and the consequent (rule) parameters are subject to optimization based on how they influence the output.
- The updates made by `update_de_do` are essential for backpropagation in neural networks or for the iterative refinement of fuzzy logic controllers, facilitating the system's training or calibration towards desired behavior or more accurate predictions.

In [ ]:
def dmf_dp(mynet, i, j):
    I = np.where(mynet['config'][:, i] == 1)[0]
    x = mynet['nodes'][I]
    a = mynet['mparams'][i - mynet['ni'], 0]
    b = mynet['mparams'][i - mynet['ni'], 1]
    c = mynet['mparams'][i - mynet['ni'], 2]
    tmp1 = (x - c) / a
    if tmp1 == 0:
        tmp2 = 0
    else:
        tmp2 = (tmp1**2)**b
    denom = (1 + tmp2) * (1 + tmp2)
    if j == 0:
        tmp = (2 * b * tmp2) / (a * denom)
    elif j == 1 and tmp1 == 0:
        tmp = 0
    elif j == 1 and tmp1 != 0:
        tmp = (-np.log(tmp1**2) * tmp2) / denom
    elif j == 2 and x == c:
        tmp = 0
    elif j == 2 and x != c:
        tmp = (2 * b * tmp2) / ((x - c) * denom)
    return tmp

The `dmf_dp` function calculates the partial derivative of a membership function's output with respect to its parameters, within the context of a system represented by the `mynet` dictionary. This function is crucial for gradient-based optimization processes, especially in systems that use fuzzy logic, such as fuzzy neural networks or fuzzy logic controllers.

### Inputs and Variables:

- **mynet**: A dictionary that encapsulates the system's configuration, including the connectivity (`'config'`), node values (`'nodes'`), and parameters of membership functions (`'mparams'`).
- **i**: The index of a specific node or membership function for which the derivative is being calculated.
- **j**: The parameter index (0 for `a`, 1 for `b`, 2 for `c`) of the membership function for which the derivative is being calculated.

### Process Explained:

1. **Identifying Connected Nodes**: 
   - It finds all nodes that are connected to node `i` using the network's configuration (`'config'`), presumably to determine the input value `x` that affects the membership function associated with node `i`.

2. **Retrieving Membership Function Parameters**:
   - Retrieves the parameters (`a`, `b`, `c`) of the membership function associated with node `i`. These parameters typically define the shape and position of the membership function curve.

3. **Calculating the Membership Function's Contribution**:
   - Calculates an intermediate value `tmp1` as `(x - c) / a`, which represents the normalized distance of the input `x` from the center `c` of the membership function, scaled by its width `a`.
   - Then, it computes `tmp2` based on `tmp1`, which is used in calculating the derivative. This involves raising `tmp1` to the power of `2b`, indicating a non-linear relationship that depends on the parameter `b`.

4. **Derivative Calculation**:
   - The derivative of the membership function's output with respect to its parameters (`a`, `b`, `c`) is computed differently based on the parameter being considered (`j`).
     - **For `a` (j == 0)**: The derivative involves `tmp2`, the parameter `b`, and the width `a` of the membership function, considering the influence of the width on the function's output.
     - **For `b` (j == 1)**: If `tmp1` is not zero, the derivative is computed using the logarithm of `tmp1^2`, showing how changes in the slope parameter `b` affect the output. If `tmp1` is zero, the derivative is zero.
     - **For `c` (j == 2)**: The derivative depends on the position `c` relative to the input `x`, indicating how shifts in the center of the membership function influence its output. If `x` is equal to `c`, the derivative is zero, reflecting no change in output due to shifts at this point.

### Purpose:

The `dmf_dp` function's ability to calculate these derivatives is essential for optimizing the parameters of membership functions in fuzzy systems. By knowing how small changes in the parameters `a`, `b`, and `c` affect the output, the system can adjust these parameters to improve performance, such as minimizing error in a predictive model or enhancing control in a fuzzy logic controller.

This detailed approach to calculating derivatives reflects the complex nature of fuzzy systems, where the shape and position of membership functions directly influence the system's behavior and outcomes.

In [ ]:
def dconsequent_dp(mynet, i, j):
    wn = mynet['nodes'][i - mynet['nc']-1]
    inp = mynet['nodes'][:mynet['ni']]
    inp = np.append(inp, 1)
    return wn * inp[j]

The `dconsequent_dp` function calculates the partial derivative of a network's output with respect to its consequent parameters. This function is used within a computational model or system represented by the `mynet` dictionary, likely part of a fuzzy logic controller or a neural network that employs rules or connections between nodes for decision-making or prediction.

### Explanation of the Code:

- **Parameters:**
  - `mynet`: A dictionary containing the system's configuration, including node values (`'nodes'`) and possibly other parameters like the number of inputs (`'ni'`) and connections or rules (`'nc'`).
  - `i`: The index of a specific node in the network, likely corresponding to a rule's consequent or a neuron's output in the context of neural networks.
  - `j`: The index of a parameter for which the derivative is being calculated. 

- **Process:**
  - **Weight Extraction (`wn`):** The function begins by calculating an offset index (`i - mynet['nc']-1`) to find the weight (`wn`) associated with the `i`th node. This weight is presumably a part of the network's structure that influences the output based on the rule's or neuron's input.
  
  - **Input Preparation (`inp`):** It extracts all inputs to the network (`mynet['nodes'][:mynet['ni']]`), representing the data fed into the system up to the point of the number of inputs (`'ni'`). Then, it appends `1` to this array, likely to account for a bias term in the calculation, making `inp` a vector of input values plus a bias.
  
  - **Partial Derivative Calculation:** The function calculates the partial derivative of the network's output with respect to the parameter at index `j` by multiplying the weight (`wn`) by the value of the input (or bias, if `j` corresponds to the appended `1`) at index `j`. This calculation reflects the contribution of each input (and the bias) to the output, scaled by the weight (`wn`).

### Purpose and Context:

The `dconsequent_dp` function is crucial for understanding and optimizing how each input (and bias) affects the output of a rule or neuron within the system. By calculating the partial derivatives of the output with respect to each input parameter and bias, the system can adjust these parameters to minimize error or optimize performance, following the principles of gradient descent or similar optimization algorithms.

This approach is commonly used in the training of neural networks and in tuning fuzzy logic controllers, where it's essential to know how changes in inputs and rule parameters affect the system's output. The inclusion of a bias term in the calculations allows for more flexible modeling, accounting for constant offsets in the output that are not directly proportional to any input.

In [ ]:
def update_step_size(mynet, error_array, iter, step_size, decrease_rate, increase_rate):
    if check_decrease_ss(error_array, mynet['last_decrease_ss'], iter):
        step_size = step_size * decrease_rate
        mynet['last_decrease_ss'] = iter
    elif check_increase_ss(error_array, mynet['last_increase_ss'], iter):
        step_size = step_size * increase_rate
        mynet['last_increase_ss'] = iter

    return mynet, step_size

The `update_step_size` function dynamically adjusts the step size (also known as the learning rate in machine learning contexts) used in an optimization process based on recent error trends. It's designed to interact with a network or model represented by `mynet`, which is part of an iterative learning or optimization algorithm. This approach helps in adapting the step size for more efficient learning or convergence to a solution.

### Parameters:

- **mynet**: A dictionary representing the state and configuration of the model or network. It stores metadata like `last_decrease_ss` and `last_increase_ss`, which track the iterations at which the last step size adjustments were made.
- **error_array**: An array or list containing the history of error values. The function uses this to decide whether to adjust the step size.
- **iter**: The current iteration number of the optimization or learning process.
- **step_size**: The current step size or learning rate.
- **decrease_rate**: A multiplier less than 1 to decrease the step size.
- **increase_rate**: A multiplier greater than 1 to increase the step size.

### Process Explained:

1. **Check for Step Size Decrease**:
   - The function first calls `check_decrease_ss` with the current `error_array`, `mynet['last_decrease_ss']`, and the current iteration number. This likely checks if the error trend suggests the step size should be decreased (e.g., if the error is oscillating or increasing, indicating too large a step size).
   - If this condition is true, the step size is multiplied by `decrease_rate` to reduce it, helping to stabilize the learning process. The iteration number at which this decrease occurs is recorded in `mynet['last_decrease_ss']`.

2. **Check for Step Size Increase**:
   - If the step size is not decreased, the function then checks whether it should be increased by calling `check_increase_ss`. This might check for conditions indicating that the learning process could safely accelerate, such as a consistent decrease in error.
   - If the criteria are met, the step size is multiplied by `increase_rate`, making it larger, to potentially speed up convergence. The iteration number of this adjustment is stored in `mynet['last_increase_ss']`.

### Output:

- The function returns the updated `mynet` dictionary (with potentially updated `last_decrease_ss` or `last_increase_ss` values) and the adjusted `step_size`.

### Purpose and Context:

Adaptive step size adjustment is a critical feature in many optimization algorithms, especially in the context of training neural networks or other machine learning models. By adjusting the step size based on recent performance (as measured by error trends), the algorithm can become more robust to the choice of initial step size, avoid oscillations or divergences, and potentially converge more quickly to a solution.

This dynamic adjustment strategy allows for a balance between the speed of convergence (larger step sizes) and the stability or accuracy of the learning process (smaller step sizes), adapting to the landscape of the error function as learning progresses.

In [ ]:
def check_decrease_ss(error_array, last_change, current):
    if current - last_change < 4:
        return False
    elif (error_array[current] < error_array[current - 1] and
          error_array[current - 1] > error_array[current - 2] and
          error_array[current - 2] < error_array[current - 3] and
          error_array[current - 3] > error_array[current - 4]):
        return True
    else:
        return False

The `check_decrease_ss` function is designed to decide whether to decrease the step size (learning rate) in an iterative optimization process, such as gradient descent in machine learning training, based on the recent trend of errors. It aims to identify a specific pattern in the error rates that suggests a potential improvement by decreasing the step size.

### Parameters:

- **error_array**: An array (or list) of error values recorded over iterations.
- **last_change**: The iteration number at which the last step size change (increase or decrease) occurred.
- **current**: The current iteration number.

### Process Explained:

1. **Check Time Since Last Change**: 
   - The function first checks if the current iteration is at least 4 iterations away from the last change in step size (`current - last_change < 4`). This condition ensures that any change to the step size is given enough time to manifest its effects on the optimization process before considering another adjustment. If this condition is not met, it returns `False`, indicating no decrease in step size should occur.

2. **Pattern Recognition in Errors**:
   - Next, it checks for a specific pattern in the last five error values, including the current iteration's error. The pattern it looks for is a 'zigzag' or oscillatory pattern where the errors alternatively increase and decrease:
     - The error decreased from two iterations ago to the previous iteration (`error_array[current - 2] < error_array[current - 3]`).
     - Then it increased from the previous iteration to the current iteration (`error_array[current] < error_array[current - 1]`).
     - Before this increase, it had decreased from three iterations ago to two iterations ago.
     - And before this decrease, it had increased from four iterations ago to three iterations ago.
   - This pattern suggests that the optimization might be oscillating around a minimum, possibly due to a step size that's too large, making it unable to settle into the minimum efficiently.

3. **Decision to Decrease Step Size**:
   - If the specified pattern is detected, the function returns `True`, suggesting that decreasing the step size could help stabilize the optimization process and potentially lead to better convergence by reducing the oscillations.
   - If the pattern is not detected, it returns `False`, indicating that the criteria for decreasing the step size were not met based on the recent error trends.

### Purpose:

This function's logic is based on the idea that if the optimization process exhibits an oscillatory behavior, reducing the step size might help in making more precise adjustments to approach the minimum of the error function more smoothly. This approach is part of adaptive learning rate strategies, which aim to dynamically adjust the learning rate based on the behavior of the error over iterations to improve the efficiency and effectiveness of the optimization process.

In [ ]:
def check_increase_ss(error_array, last_change, current):
    if current - last_change < 4:
        return False
    elif (error_array[current] < error_array[current - 1] and
          error_array[current - 1] < error_array[current - 2] and
          error_array[current - 2] < error_array[current - 3] and
          error_array[current - 3] < error_array[current - 4]):
        return True
    else:
        return False

The `check_increase_ss` function evaluates whether the step size (or learning rate) in an optimization process should be increased, based on the trend of errors over recent iterations. This function is particularly relevant in contexts like machine learning model training, where adjusting the learning rate can significantly impact the efficiency and outcome of the training process.

### Parameters:

- **error_array**: An array (or list) containing the error values observed over a series of iterations.
- **last_change**: The iteration number at which the last adjustment (increase or decrease) to the step size was made.
- **current**: The current iteration number being evaluated.

### Logic and Process:

1. **Check Interval Since Last Adjustment**: 
   - Initially, the function ensures that a sufficient number of iterations (`4` in this case) have passed since the last adjustment to the step size. This is to allow enough time for previous adjustments to manifest their effects on the optimization process. If fewer than 4 iterations have passed (`current - last_change < 4`), it returns `False`, indicating that it's too soon to make another adjustment.

2. **Evaluating the Trend of Errors**:
   - The core of the function checks for a continuous decrease in error over the last five iterations, including the current one. Specifically, it looks for a scenario where each iteration's error is less than the error of the iteration before it (`error_array[current] < error_array[current - 1]` and so on, up to `error_array[current - 3] < error_array[current - 4]`). This pattern suggests a consistent improvement in the optimization process over these iterations.

3. **Decision on Increasing Step Size**:
   - If the decreasing trend in errors is confirmed, the function returns `True`, indicating that increasing the step size could be beneficial. The rationale is that if errors are consistently decreasing, a larger step size might accelerate convergence towards the minimum of the error function without risking overshooting or oscillation.
   - If the decreasing trend is not present, it returns `False`, suggesting that the criteria for safely increasing the step size are not met based on the recent error trends.

### Purpose and Context:

`check_increase_ss` is used within adaptive learning rate strategies to dynamically adjust the step size based on the observed performance of the optimization process. By identifying a situation where the error consistently decreases, it infers that the optimization process may be proceeding too cautiously and that a larger step size could expedite convergence. This approach helps balance the trade-off between the speed of convergence and the risk of missing the minimum due to too large a step size.

In [ ]:
def update_parameter(mynet, step_size):
    tmp = mynet['mparam_de_do']
    tmp = tmp * tmp
    norm_factor = np.sqrt(np.sum(tmp))
    mynet['mparams'] = mynet['mparams'] - step_size * mynet['mparam_de_do'] / norm_factor
    return mynet

The `update_parameter` function is designed to update the parameters of the membership functions (`'mparams'`) in a model or network (`mynet`) using gradient descent with an added normalization step. This function adjusts the parameters based on the gradients stored in `'mparam_de_do'`, scaled by a given `step_size` and normalized by the magnitude of these gradients. The purpose of this approach is to make the update step size-independent of the gradient magnitude, potentially leading to more stable and efficient optimization.

### Steps and Logic:

1. **Gradient Squaring and Summation**:
   - `tmp = mynet['mparam_de_do']`: Retrieves the gradient of the error with respect to the membership function parameters (`'mparam_de_do'`).
   - `tmp = tmp * tmp`: Squares each element of the gradient matrix. This operation prepares for computing the norm (magnitude) of the gradient vector by squaring each component.
   - `norm_factor = np.sqrt(np.sum(tmp))`: Calculates the norm of the gradient vector by summing the squared components and then taking the square root. This norm represents the magnitude of the gradient vector and is used for normalization.

2. **Parameter Update with Normalization**:
   - The membership function parameters (`'mparams'`) are updated by subtracting a fraction of the gradient vector from the current parameters. The fraction is determined by the product of the `step_size` and the normalized gradient vector (`mynet['mparam_de_do'] / norm_factor`).
   - Normalizing the gradient (`mynet['mparam_de_do'] / norm_factor`) ensures that the update step is proportional to the direction of the gradient but independent of its magnitude. This can help prevent overly large updates that might occur with large gradients, contributing to more stable convergence.

3. **Return Updated Model**:
   - Finally, the function returns the updated `mynet` dictionary with the new values of `'mparams'`.

### Purpose and Context:

This function is a critical component of the training process for models that involve membership functions, likely in the context of fuzzy logic systems or hybrid models that incorporate fuzzy logic components. By updating the membership function parameters in the direction that reduces error (as indicated by the negative gradient) and normalizing this update by the gradient's magnitude, the function aims to achieve efficient and stable optimization.

The inclusion of normalization is particularly noteworthy, as it addresses the challenge of varying gradient magnitudes across different parameters or iterations, helping to ensure that each update step is appropriately scaled for consistent progress towards the optimization goal.

In [ ]:
def evalmyanfis(mynet, inputs):
    anfis_output = np.zeros((inputs.shape[0], 1))
    for i in range(inputs.shape[0]):
        mynet['nodes'][:mynet['ni']] = inputs[i]
        # mynet['nodes'][:mynet['ni']] = inputs[j].T.tolist()
        mynet = calculate_output1(mynet)
        mynet = calculate_output2(mynet)
        mynet = calculate_output3(mynet)
        mynet = calculate_output4(mynet)
        mynet = calculate_output5(mynet)

        anfis_output[i, 0] = mynet['nodes'][-1]
    
    return anfis_output

The `evalmyanfis` function evaluates the outputs of an Adaptive Neuro-Fuzzy Inference System (ANFIS) given a set of inputs. ANFIS combines neural network learning capabilities with fuzzy logic to model complex nonlinear functions. This function iterates over each input vector, processes it through the network's layers, and collects the output.

### Understanding the Components:

- **mynet**: A dictionary that represents the ANFIS model. It includes various elements like the number of inputs (`'ni'`), the network nodes (`'nodes'`), and possibly others related to the fuzzy system's rules and membership functions.

- **inputs**: A NumPy array of input vectors. Each row represents a different input vector to be evaluated by the ANFIS model.

### Process Explained:

1. **Initialize Output Array**:
   - `anfis_output` is initialized as a zeros array with the same number of rows as the `inputs` array and one column. This will store the ANFIS output for each input vector.

2. **Iterate Over Input Vectors**:
   - The function iterates over each input vector (`inputs[i]`). For each vector:
     - It assigns the input vector to the beginning of the `'nodes'` array in `mynet`. This step sets the input layer of the ANFIS model for the current evaluation.

3. **Process Through ANFIS Layers**:
   - The input vector is then processed through several stages of the ANFIS model, represented by calls to `calculate_output1`, `calculate_output2`, `calculate_output3`, `calculate_output4`, and `calculate_output5`. These functions presumably perform various operations such as applying membership functions, rule evaluation, and aggregation, corresponding to how an ANFIS model processes inputs to generate an output.

4. **Collect the Output**:
   - After the input vector has been processed through all stages, the final output is stored in `anfis_output[i, 0]`. This output is taken from the last element of the `'nodes'` array in `mynet`, which represents the final result of the ANFIS model for the given input vector.

5. **Return the Outputs**:
   - Once all input vectors have been processed, the function returns the `anfis_output` array containing the ANFIS outputs for each input.

### Purpose and Use:

This function is used to evaluate the performance of an ANFIS model on a set of input vectors, generating outputs that can be compared to actual values to assess accuracy, or used for predictions in practical applications. The stepwise processing through different ANFIS components (represented by `calculate_outputX` functions) reflects the layered and complex nature of neuro-fuzzy systems, which combine aspects of fuzzy logic (for handling uncertainty and imprecision) with neural networks (for learning from data).

In [ ]:
def myanfis(data, inputs, epoch_n, mf, step_size, decrease_rate, increase_rate):
    # Divide data into input and output    
    output = data[:, -1:]  # The last column is the output

    ndata = data.shape[0]  # Data length

    # Define minimum and maximum of input to determine initial membership functions
    mn = np.min(inputs, axis=0)
    mx = np.max(inputs, axis=0)
    mm = mx - mn
    ni = inputs.shape[1]  # Number of inputs
    nc = mf**ni  # Number of rules
    Node_n = ni + ni * mf + 3 * nc + 1  # Total number of nodes
    
    min_RMSE = 999999999999  # Define minimum RMSE

    # Define initial membership functions
    mparams = []
    for i in range(ni):
        tmp = np.linspace(mn[i], mx[i], mf)
        mparams.extend(np.column_stack([np.full(mf, mm[i] / 6), np.full(mf, 2), tmp]))

    # Define initial Kalman parameters with all zeros
    kparams = np.zeros((nc, ni + 1))
    
    # Create connection matrix and node array
    config = np.zeros((Node_n, Node_n))
    nodes = np.zeros(Node_n)

    # Inputs - layer1 connections
    st = ni
    for i in range(ni):
        config[i, st : st + mf] = 1
        st = st + mf

    # Layer1 - Layer2 connections
    st = ni + ni * mf 
    if ni == 2:
        for i in range(mf):
            for j in range(mf):
                config[ni + i, st] = 1
                config[ni + mf + j, st] = 1
                st = st + 1
    elif ni == 3:
        for i in range(mf):
            for j in range(mf):
                for k in range(mf):
                    config[ni + i, st] = 1
                    config[ni + mf + j, st] = 1
                    config[ni + 2 * mf + k, st] = 1
                    st = st + 1
    elif ni == 4:
        for i in range(mf):
            for j in range(mf):
                for k in range(mf):
                    for l in range(mf):
                        config[ni + i, st] = 1
                        config[ni + mf + j, st] = 1
                        config[ni + 2 * mf + k, st] = 1
                        config[ni + 3 * mf + l, st] = 1
                        st = st + 1
    elif ni == 5:
        for i in range(mf):
            for j in range(mf):
                for k in range(mf):
                    for l in range(mf):
                        for m in range(mf):
                            config[ni + i, st] = 1
                            config[ni + mf + j, st] = 1
                            config[ni + 2 * mf + k, st] = 1
                            config[ni + 3 * mf + l, st] = 1
                            config[ni + 4 * mf + m, st] = 1
                            st = st + 1

    # Layer2 - Layer3 connections
    for i in range(nc):
        for j in range(nc):
            config[ni + ni * mf + i, ni + ni * mf + nc + j] = 1

    # Layer3 - Layer4 connections
    for i in range(nc):
        config[ni + ni * mf + nc + i, ni + ni * mf + 2 * nc + i] = 1

    # Layer4 - Layer5 connections
    for i in range(nc):
        config[ni + ni * mf + 2 * nc + i, -1] = 1

    # Inputs - Layer4 connections
    for i in range(ni):
        for j in range(nc):
            config[i, ni + ni * mf + 2 * nc + j] = 1

    mynet = {
        "config": config,
        "mparams": np.array(mparams),
        "kparams": kparams,
        "nodes": nodes,
        "ni": ni,
        "mf": mf,
        "nc": nc,
        "last_decrease_ss": 1,
        "last_increase_ss": 1,
    }
    
    num_nodes = len(mynet['nodes'])  # You need to determine this
    
    layer_1_to_3_output = np.zeros((num_nodes, ndata))  # Initialize with the correct shape
    

    for iter in range(0, epoch_n):
        for j in range(ndata):

            mynet['nodes'] = mynet['nodes'].flatten()
            mynet['nodes'][:mynet['ni']] = inputs[j]

            mynet = calculate_output1(mynet)
            mynet = calculate_output2(mynet)
            mynet = calculate_output3(mynet)
                     
            # Save outputs of layer 1 to 3
            layer_1_to_3_output[:, j] = mynet['nodes'].flatten() 
            # print((mynet['nodes'].flatten()))
            # print((layer_1_to_3_output))
            # exit()   
            kalman_data = get_kalman_data(mynet, output[j])
            
            # exit()
            mynet = mykalman(mynet, kalman_data, j)
            
        mynet = clear_de_dp(mynet)
    
        # print(np.shape(layer_1_to_3_output))
        # print((layer_1_to_3_output))
        # file_path = "layer_1_to_3_output.csv"

        ## Use numpy.savetxt to save the array to a CSV file
        # np.savetxt(file_path, layer_1_to_3_output, delimiter=',')
        # exit()
        anfis_output = np.zeros((ndata, 1))
        RMSE = np.zeros((epoch_n, 1))  # Initialize RMSE as a NumPy array of zeros

        for j in range(ndata):
            mynet['nodes'] = layer_1_to_3_output[:, j]   
            
            
            mynet = calculate_output4(mynet)
            mynet = calculate_output5(mynet)
        
            anfis_output[j, 0] = mynet['nodes'][-1]
            target = output[j]
            
            de_dout = -2 * (target - anfis_output[j, 0])
        
            mynet = calculate_de_do(mynet, de_dout)
            
            mynet = update_de_do(mynet)
            

        diff = anfis_output - output
        total_squared_error = np.sum(diff**2)
        RMSE[iter - 1, 0] = np.sqrt(total_squared_error / ndata)
        print(f'{iter}. RMSE error: {RMSE[iter - 1, 0]}')
        # input()
        if RMSE[iter - 1, 0] < min_RMSE:
            bestnet = mynet
            min_RMSE = RMSE[iter - 1, 0]

        mynet = update_parameter(mynet, step_size)
        mynet, step_size = update_step_size(mynet, RMSE, iter, step_size, decrease_rate, increase_rate)

# At this point, bestnet contains the trained ANFIS model, and RMSE contains the RMSE values for each epoch.
    for j in range(ndata):       
        # print(j)
        # mynet['nodes'][:mynet['ni']].flatten()
        # print(np.shape(mynet['nodes'][:mynet['ni']]))
        # print(np.shape(inputs[j]))
        
        mynet['nodes'][:mynet['ni']] = inputs[j]
        # mynet['nodes'][:mynet['ni']] = inputs[j].T.tolist()
        mynet = calculate_output1(mynet)
        mynet = calculate_output2(mynet)
        mynet = calculate_output3(mynet)
        mynet = calculate_output4(mynet)
        mynet = calculate_output5(mynet)

        anfis_output[j, 0] = mynet['nodes'][-1]
    
    return bestnet,anfis_output,RMSE

The `myanfis` function is designed to train an Adaptive Neuro-Fuzzy Inference System (ANFIS) model using a dataset. This complex function integrates various stages of the ANFIS architecture, dynamically adjusting model parameters based on input data, and employs a version of the Kalman filter for optimization. The goal is to minimize the Root Mean Square Error (RMSE) between the model's predictions and actual outputs over a specified number of epochs.

### Breakdown of the Function:

#### Initialization and Setup:
- **Data Division**: Separates the provided dataset into inputs and outputs.
- **Parameter Initialization**: Establishes the initial settings based on the inputs, such as the range for membership functions and the structure of the network (e.g., number of rules, total nodes).
- **Membership Functions**: Initializes membership function parameters (`mparams`) using evenly spaced values between the minimum and maximum of each input feature.
- **Kalman Parameters and Network Configuration**: Sets up the initial values for the Kalman filter parameters (`kparams`) and the connection matrix (`config`) defining the ANFIS structure.

#### Main Training Loop:
- Iterates over each epoch:
  - For each data point:
    - **Feeds Input Data**: Sets the current input data into the model.
    - **Forward Pass**: Processes the input through several layers/functions (`calculate_output1` to `calculate_output5`) to generate predictions.
    - **Kalman Data Preparation and Optimization**: Prepares data for the Kalman filter (`get_kalman_data`), applies the Kalman filter (`mykalman`) for parameter optimization, and clears derivative information (`clear_de_dp`).
  - **Error Calculation and Parameter Update**:
    - Computes the RMSE for the epoch based on the difference between predictions and actual outputs.
    - Updates model parameters to reduce error (`calculate_de_do`, `update_de_do`, `update_parameter`).
    - Dynamically adjusts the step size for optimization (`update_step_size`).

#### Post-Training:
- **Final Model Evaluation**: After completing all epochs, it evaluates the model on the entire dataset again to capture the final predictions.
- **Model Selection**: Selects the model (`bestnet`) that achieved the lowest RMSE during training.

### Key Components:
- **ANFIS Architecture**: Mimics the layers and processing steps of an ANFIS model, including membership function evaluation, rule application, normalization, and defuzzification.
- **Dynamic Learning Rate Adjustment**: Employs a strategy to increase or decrease the learning rate based on the error trends, enhancing the training process's effectiveness.
- **Kalman Filter Optimization**: Uses a Kalman filter-based approach for fine-tuning model parameters, aiming for precise adjustments that consider the error's variance and the parameters' uncertainty.

### Usage and Output:
- The function trains the ANFIS model over the specified epochs, dynamically adjusting its parameters to minimize prediction error.
- It returns the trained model (`bestnet`), the final set of predictions (`anfis_output`), and the RMSE values for each epoch, providing a comprehensive overview of the training process and its outcomes.

This training function represents a sophisticated approach to modeling complex systems, combining fuzzy logic's interpretability with neural networks' learning capabilities, and optimizing the model parameters using advanced techniques like the Kalman filter.

In [ ]:
def plot_Nodes(mynet):
    # Plot the Node Connections
    plt.figure()
    plt.imshow(mynet['config'], aspect='auto', cmap='cool')
    plt.title('Node Connections')
    plt.show()

The `plot_Nodes` function visualizes the connections between nodes in a network (presumably an ANFIS model or a similar neural network structure) using a heatmap. This visualization is generated from the network's configuration matrix stored in `mynet['config']`, where each element in the matrix indicates the presence (or absence) of a connection between nodes.

### Breakdown of the Function:

- **`plt.figure()`**: This line initializes a new figure for plotting. It's where the upcoming plot will be drawn.

- **`plt.imshow()`**: This function displays the `mynet['config']` matrix as an image. 
  - `mynet['config']`: The configuration matrix representing the connections between nodes in the network. The matrix is expected to be binary (or possibly weighted), with a non-zero value indicating a connection between nodes and zero indicating no connection.
  - `aspect='auto'`: Sets the aspect ratio of the plot to automatic, allowing the image to adjust its dimensions to fit the array's shape without distorting the aspect ratio.
  - `cmap='cool'`: Specifies the colormap used to map scalar data to colors. The 'cool' colormap provides a visual gradient that can help distinguish between connected (non-zero values) and non-connected (zero values) nodes.
  
- **`plt.title('Node Connections')`**: Adds a title to the plot, indicating that the image represents the connections between nodes within the network.

- **`plt.show()`**: This command displays the figure. It renders the plot and shows it in a window that pops up on the screen.

### Purpose and Use:

Visualizing the node connections in a network can provide insights into the network's structure and complexity. For instance, in an ANFIS model, it might help to understand how inputs are connected to various membership functions and rules, how rules interact with each other, and how the output is aggregated from these interactions. This type of visualization can be particularly useful for debugging, designing, or explaining the model's architecture to others. 

The use of a heatmap for this purpose makes it easy to visually identify which parts of the network are densely connected and which parts are not, offering a quick overview of the network's connectivity pattern.

In [ ]:
def plot_mf(mynet, data):
    plt.figure(figsize=(12, 4))  # Create a single figure for all plots
    # print(mynet['mparams'])
    k = 0
    for i in range(0,mynet['ni']*mynet['mf'],mynet['mf']):
        # print(k)
        plt.subplot(2, 2, k+1)  # Create subplots for each input variable
        plt.title(f'Input {k+1}')
        plt.xlabel('X')
        plt.ylabel('Degree of Membership')
        min_val = np.min(data[:, k])
        max_val = np.max(data[:, k])
        step = 0.1
        x = np.arange(min_val, max_val, step)
        x = np.arange(-10, 10, step)
        
        for j in range(mynet['mf']):  
            mf_x = fuzz.gbellmf(x, mynet['mparams'][i+j][0], mynet['mparams'][i+j][1], mynet['mparams'][i+j][2])
            plt.plot(x, mf_x, label=f'Membership Function {j+1}')
        k = k+1
        plt.tight_layout()
        plt.legend()
    plt.show()

The `plot_mf` function visualizes the membership functions (MFs) defined for each input variable in a neuro-fuzzy system ANFIS model as represented by the `mynet` dictionary. This visualization aids in understanding how different input values are mapped to degrees of membership across the fuzzy sets defined for each input variable.

### Breakdown of the Function:

- **Figure Initialization**: 
  - `plt.figure(figsize=(12, 4))` initializes a new figure with a specified size for plotting all the membership functions across input variables.

- **Loop Over Input Variables**: 
  - The loop `for i in range(0, mynet['ni']*mynet['mf'], mynet['mf'])` iterates over each input variable. The iteration step is `mynet['mf']`, the number of membership functions per input, indicating that the function plots all MFs for one input variable before moving to the next.

- **Subplot Configuration**: 
  - `plt.subplot(2, 2, k+1)` creates a subplot for each input variable within the figure. The layout is set to accommodate four plots (2 rows by 2 columns), though this layout might not suit all configurations if the number of inputs differs.
  - Each subplot is titled with the input variable index (`plt.title(f'Input {k+1}')`).

- **Plotting Membership Functions**: 
  - For each input variable, the function calculates the range of `x` values based on the data or a predefined range (-10 to 10). This range is used to plot the membership functions.
  - `fuzz.gbellmf(x, mynet['mparams'][i+j][0], mynet['mparams'][i+j][1], mynet['mparams'][i+j][2])` computes the degree of membership for each point in `x` using the Gaussian bell-shaped membership function parameters stored in `mynet['mparams']`.
  - `plt.plot(x, mf_x, label=f'Membership Function {j+1}')` plots the calculated membership values against `x` for each membership function of the current input variable.

- **Final Adjustments and Plot Display**: 
  - `plt.tight_layout()` adjusts the subplots to make sure they fit well within the figure area without overlapping.
  - `plt.legend()` adds a legend to each subplot to distinguish between different membership functions.
  - `plt.show()` displays the completed figure with all subplots.

### Purpose and Use:

This function is particularly useful for analyzing and debugging an ANFIS model by visually inspecting the shape and spread of the membership functions defined for each input variable. Understanding these aspects can provide insights into how input values influence the fuzzy reasoning process, which in turn affects the model's output. It helps in assessing whether the membership functions are appropriately defined and distributed across the input space, which is crucial for the model's accuracy and generalization capability.

In [ ]:
def plot_predictions(actual_output,anfis_predictions):
    plt.figure()
    plt.plot(actual_output, 'b*', label='Actual Output')
    plt.plot(anfis_predictions, 'r-', linewidth=0.5, label='ANFIS Prediction')
    plt.xlabel('Data Point')
    plt.ylabel('Output Value')
    plt.legend()
    plt.show()

The `plot_predictions` function creates a visualization to compare actual output values against predictions made by an Adaptive Neuro-Fuzzy Inference System (ANFIS). This visual comparison helps in assessing the ANFIS model's performance in terms of how closely its predictions align with the real data.

### Breakdown of the Function:

- **Figure Initialization**:
  - `plt.figure()` initializes a new figure for plotting. This is where the actual vs. predicted outputs will be visualized.

- **Plotting Actual Outputs**:
  - `plt.plot(actual_output, 'b*', label='Actual Output')` plots the actual output values as blue stars (`'b*'`). Each star represents the true output value for a corresponding data point.

- **Plotting ANFIS Predictions**:
  - `plt.plot(anfis_predictions, 'r-', linewidth=0.5, label='ANFIS Prediction')` plots the ANFIS model's predictions as a red line (`'r-'`) with a linewidth of 0.5. This continuous line represents the predicted output values for each data point, allowing for an easy visual comparison with the actual outputs.

- **Axes Labels and Legend**:
  - `plt.xlabel('Data Point')` and `plt.ylabel('Output Value')` label the x-axis and y-axis, respectively. The x-axis represents individual data points (indexed), while the y-axis represents the output value (either actual or predicted).
  - `plt.legend()` adds a legend to the plot, distinguishing between the actual outputs and the ANFIS predictions. This is crucial for understanding which symbols/lines correspond to the actual data versus the model's predictions.

- **Displaying the Plot**:
  - `plt.show()` renders the plot and displays it in a window. This allows for a visual inspection of how well the ANFIS model's predictions match the actual outputs across all data points.

### Purpose and Use:

Visualizing actual versus predicted values is a common method to quickly assess a model's predictive accuracy. By comparing the two visually, one can identify patterns such as consistent overestimation or underestimation, periodic discrepancies, and overall alignment between the predictions and actual values. This function is particularly useful for researchers, data scientists, and developers working with ANFIS models to refine and evaluate their models based on empirical data.

In [ ]:
def calc_rmse(x,y):
    # Calculate Root Mean Squared Error (RMSE)
    mse = mean_squared_error(x, y)
    rmse = np.sqrt(mse)
    
    return rmse

In [ ]:
def calc_r2(x,y):
    # Calculate R-squared
    r_squared = r2_score(x, y)
    
    return r_squared

In [ ]:
def plot_r2(x,y):
    
    r_squared = r2_score(x, y)
     # Create a scatter plot for the data points
    plt.figure(figsize=(8, 6))
    plt.scatter(x, y, label=f'R-squared = {r_squared:.4f}')
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)  # Add the identity line
    plt.xlabel('Actual Output')
    plt.ylabel('Predicted Output')
    plt.title("Correlation_Coefficient (R2)")
    plt.legend(loc='upper left')
    plt.tight_layout()
    # Show the plot
    plt.show